<a target="_blank" href="https://colab.research.google.com/github/univiemops/tewa1-computational-cognition/blob/main/extra/03%20Recap.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


Recap lab 3
===

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

treatment = np.array(
    [
        444.48703626,
        420.71413104,
        482.04432447,
        380.46896668,
        420.56864234,
        474.09130417,
        414.9748433,
        450.15423802,
        436.53977461,
        500.12705411,
        405.00705696,
        419.3141794,
        460.46096974,
        450.54358948,
        420.93431563,
        467.40481135,
        510.84094939,
        482.61924772,
        480.32638462,
        510.76756724,
    ]
)

control = np.array(
    [
        420.1243685,
        501.25211241,
        454.37132587,
        600.39850065,
        501.79657108,
        481.94197109,
        469.51703441,
        449.82747137,
        450.98838458,
        477.15878941,
        570.00039675,
        460.18766471,
        432.70480616,
        480.38394358,
        478.46070285,
        485.71067427,
        487.91937261,
        505.86604195,
        495.8480102,
        480.9547509,
    ]
)

## 1. Exercise: Bootstrapped confidence intervals

1. Calculate the bootstrapped confidence intervals for the mean differences in the treatment and control groups with 2000 bootstraps. With the obtained 2000 resampled mean differences, use the `np.percentile` function to find the 95% confidence intervals, and store them as the array `ci`.
2. Visualize the mean differences using a histogram and indicate the confidence interval with 2 lines.

### 1.1. Calcuate bootstrapped confidence intervals

In [ ]:
n_bootstraps = 2000
sample_size = len(treatment)

np.random.seed(0)


def bootstrap(arr, sample_size, n_bootstraps, replace=True):

    means = list()
    for i in range(n_bootstraps):
        mean_i = np.random.choice(arr, sample_size, replace=replace).mean()
        means.append(mean_i)

    return np.array(means)


def confidence_intervals(arr, alpha=0.05):

    ci_lower = np.percentile(arr, 100 * alpha / 2)
    ci_upper = np.percentile(arr, 100 * 1 - (alpha / 2))

    return ci_lower, ci_upper


means_treatment_bs = bootstrap(treatment, 20, 2000)
means_control_bs = bootstrap(control, 20, 2000)
mean_diff_bs = means_treatment_bs - means_control_bs

ci = confidence_intervals(mean_diff_bs, 0.05)
print(ci)

### 1.2. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(mean_diff_bs, bins=100, label="2,000 sample mean diffs")
ax.axvline(
    ci[0], color="lightgrey", ls="--", label=f"95% CI ({ci[0]:+.2f}, {ci[1]:+.2f})"
)
ax.axvline(ci[1], color="lightgrey", ls="--")
ax.legend(loc="upper right");

## 2. Exercise: Comparing $t$-test and permutation test
Imagine you are doing a reaction time study in a rare patient group and you are having trouble finding patients, so you end up with only 5 patients. Obviously, it is easier to get many healthy people and you have 50 participants in your control group. Could you do the following tasks?
1. Randomly select 5 participants from a normal distribution with a mean of 600 and a sd of 50, call it the variable `patient`; randomly select 50 participants from a normal distribution with a mean of 500 and a sd of 100, call it the variable `non_patient`.
2. Run the $t$-test on these two groups.
3. Run the permutation test on these two groups to compare the differences of means.
4. Repeat tasks 1-3 500 times and save the p-values from the two tests as arrays named `p_t_test` and `p_perm_test`. 
5. Choose a way to visualize `p_t_test` and `p_perm_test` to compare the difference/similarity between them.

### 2.1. Randomly select 5 participants ... 
... from a normal distribution with a mean of 600 and a sd of 50, call it the variable `patient`; randomly select 50 participants from a normal distribution with a mean of 500 and a sd of 100, call it the variable `non_patient`.

In [ ]:
np.random.seed(0)

patient = np.random.normal(600, 50, 5)
non_patient = np.random.normal(500, 100, 50)

print("mean difference:", patient.mean() - non_patient.mean())

### 2.2. Run the t-test on these two groups

In [ ]:
print(
    "p-value t-test:",
    stats.ttest_ind(patient, non_patient, alternative="two-sided").pvalue,
)

### 2.3. Run the permutation test ...
... on these two groups to compare the differences of means.

In [ ]:
# 3


def statistic(x, y):
    return np.mean(x) - np.mean(y)


perm_test = stats.permutation_test(
    [patient, non_patient],
    statistic,
    permutation_type="independent",
    n_resamples=1000,
    alternative="two-sided",
)

print("p-value permutation test:", perm_test.pvalue)

### 2.4. Repeat tasks 1-3 500 times and ...
... save the p-values from the two tests as arrays named `p_t_test` and `p_perm_test`. 

In [ ]:
n_sims = 500

p_t_test = np.ones(n_sims)
p_perm_test = np.ones(n_sims)

for i in range(n_sims):

    patient = np.random.normal(600, 50, 5)
    non_patient = np.random.normal(500, 100, 50)

    p_t_test[i] = stats.ttest_ind(patient, non_patient, alternative="two-sided").pvalue
    p_perm_test[i] = stats.permutation_test(
        [patient, non_patient],
        statistic,
        permutation_type="independent",
        n_resamples=1000,
        alternative="two-sided",
    ).pvalue

print("significant proportion t-test:", np.sum([p_t_test < 0.05]) / n_sims)
print("significant proportion perm. test:", np.sum([p_perm_test < 0.05]) / n_sims)

### 2.5. Choose a way to visualize ...
... `p_t_test` and `p_perm_test` to compare the difference/similarity between them.

In [ ]:
print("r =", np.corrcoef(p_t_test, p_perm_test)[0][1])

fig = plt.figure(figsize=(6, 6))
plt.scatter(p_t_test, p_perm_test, alpha=0.2)
plt.xlabel("p-value t-test")
plt.ylabel("p-value permutationtest");

Alternative (but there are many ways ...)

In [ ]:
fig = plt.figure(figsize=(12, 6))

plt.plot(
    np.arange(n_sims),
    p_t_test - p_perm_test,
    alpha=1.0,
    color="steelblue",
)
plt.title(f"{n_sims} simulations")
plt.xlabel("simulations")
plt.ylabel(rf"$\Delta$ p-value");